# 08 · Frozen grouped nested evaluation

Execute the full nested procedure **only from a verified freeze**. Notebook 07 is immutable preparation evidence; this notebook adds a separately hashed runner without changing candidates, architecture, folds or selection rules. [Runner guide](../docs/nested-evaluation-runner.md).

The default real-data action is `verify`, which fits no models. Set `APNEA_NESTED_ACTION=full` for a real search, or `APNEA_PORTFOLIO_SMOKE=1` and `APNEA_NESTED_ACTION=smoke` for the synthetic end-to-end test. `APNEA_NESTED_TAG` selects a fresh output/resume identity. Use the same `APNEA_PORTFOLIO_RUN` as Notebooks 05–07. A mismatched source, dependency, input or checkpoint fails rather than being silently reused.

In [1]:
import ast
import fcntl
import hashlib
import importlib.metadata
import io
import json
import os
import pickle
import platform
import re
import subprocess
import sys
import tempfile
import warnings
from contextlib import contextmanager
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    average_precision_score,
)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())


def root_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


SMOKE = os.environ.get("APNEA_PORTFOLIO_SMOKE") == "1"
RUN = root_path(
    os.environ.get("APNEA_PORTFOLIO_RUN", "data/processed/portfolio-v2/plan001-real-20261108")
)
PHASE1 = root_path(os.environ.get("APNEA_RUN", "data/processed/phase1"))
FEATURE_RUN = root_path(os.environ.get("APNEA_PHASE2_RUN", "data/processed/phase2-grouped"))
ACTION = os.environ.get("APNEA_NESTED_ACTION", "smoke" if SMOKE else "verify")
TAG = os.environ.get("APNEA_NESTED_TAG", "runner-v1")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", TAG) or TAG in {".", ".."}:
    raise ValueError("Invalid run tag")
if ACTION not in {"verify", "smoke", "full", "summarize"}:
    raise ValueError("Unknown evaluation action")
if (ACTION == "smoke" and not SMOKE) or (ACTION == "full" and SMOKE):
    raise ValueError("Wrong smoke/real action")
torch.set_num_threads(4)

## 1. Frozen dependency validation, task integrity and saved folds

Only whitelisted definitions are imported from notebooks; no download, training or control cell is executed as a dependency. Native XGBoost operations stay isolated from PyTorch. Stage locks prevent concurrent writers and release automatically on process exit. Each task is atomic, binds the freeze and runner hashes, and logs completion/resume/failure. Only locally produced, hash-verified checkpoint pickle files are loaded.

In [2]:
def install_dependencies(root, namespace):
    """Load only named definitions; never execute another notebook's workflow cells."""
    import random as random_module

    import xgboost as xgb_module
    from sklearn import (
        ensemble,
        impute,
        linear_model,
        metrics,
        model_selection,
        pipeline,
        preprocessing,
        svm,
    )

    namespace.update(
        {
            "random": random_module,
            "RandomForestClassifier": ensemble.RandomForestClassifier,
            "SimpleImputer": impute.SimpleImputer,
            "LogisticRegression": linear_model.LogisticRegression,
            "SVC": svm.SVC,
            "XGBClassifier": xgb_module.XGBClassifier,
            "StandardScaler": preprocessing.StandardScaler,
            "make_pipeline": pipeline.make_pipeline,
            "StratifiedGroupKFold": model_selection.StratifiedGroupKFold,
            **{
                name: getattr(metrics, name)
                for name in [
                    "accuracy_score",
                    "average_precision_score",
                    "confusion_matrix",
                    "f1_score",
                    "precision_recall_curve",
                    "precision_score",
                    "roc_auc_score",
                ]
            },
        }
    )
    groups = {
        "05_evaluation_protocol.ipynb": [
            "choose_threshold",
            "validate_predictions",
            "metric_values",
            "summarize_predictions",
            "group_weights",
            "paired_group_bootstrap",
            "validate_cohort",
            "require_classes",
            "build_fold_assignments",
            "validate_fold_assignments",
        ],
        "06_literature_baselines.ipynb": ["constant_references", "literature_feature_names"],
        "07_nested_model_comparison.ipynb": [
            "make_cnn",
            "normalize_minute",
            "make_classical_model",
            "reset_seed",
            "train_epoch",
        ],
    }
    for name, wanted in groups.items():
        notebook = json.loads((Path(root) / "notebooks" / name).read_text())
        nodes = []
        for cell in notebook["cells"]:
            if cell["cell_type"] == "code":
                nodes.extend(
                    n
                    for n in ast.parse("".join(cell["source"])).body
                    if isinstance(n, ast.FunctionDef) and n.name in wanted
                )
        if {n.name for n in nodes} != set(wanted):
            raise ValueError(f"Missing or duplicated dependency definitions: {name}")
        exec(
            compile(ast.Module(body=nodes, type_ignores=[]), f"<frozen:{name}>", "exec"), namespace
        )


def canonical(value):
    return json.dumps(
        value,
        sort_keys=True,
        allow_nan=False,
        default=lambda x: x.item() if isinstance(x, np.generic) else str(x),
    )


def digest_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def verify_freeze(run, root, smoke):
    run, root = Path(run), Path(root)
    payload = json.loads((run / "freeze/payload.json").read_text())
    manifest = json.loads((run / "freeze/manifest.json").read_text())
    if digest_file(run / "freeze/payload.json") != manifest["payload_hash"]:
        raise ValueError("Frozen payload hash mismatch")
    if (
        hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
        != manifest["identity_hash"]
    ):
        raise ValueError("Frozen identity mismatch")
    identity = payload["identity"]
    if identity["synthetic"] != smoke or payload["status"] != (
        "frozen_synthetic" if smoke else "frozen"
    ):
        raise ValueError("Smoke/real freeze mismatch")
    for name, expected in identity["source_hashes"].items():
        if digest_file(root / "notebooks" / name) != expected:
            raise ValueError(f"Frozen source drift: {name}")
    if digest_file(root / "uv.lock") != identity["lock_sha256"]:
        raise ValueError("Dependency lock drift")
    for package, version in identity["versions"].items():
        if importlib.metadata.version(package) != version:
            raise ValueError(f"Package drift: {package}")
    if (
        platform.platform() != identity["platform"]
        or platform.python_version() != identity["python"]
    ):
        raise ValueError("Platform/Python drift: document a pre-evaluation amendment")
    for stage, expected_id in [
        ("protocol", identity["protocol_identity"]),
        ("baselines", identity["baseline_identity"]),
    ]:
        metadata = json.loads((run / stage / "manifest.json").read_text())
        if stage == "protocol":
            fields = {k: metadata[k] for k in ["configuration", "sources", "environment", "python"]}
        else:
            fields = {k: v for k, v in metadata.items() if k not in ["identity_hash", "artifacts"]}
        if (
            metadata["identity_hash"] != expected_id
            or hashlib.sha256(json.dumps(fields, sort_keys=True).encode()).hexdigest()
            != expected_id
        ):
            raise ValueError(f"{stage} identity drift")
        for name, expected in metadata["artifacts"].items():
            if digest_file(run / stage / name) != expected:
                raise ValueError(f"{stage} artifact drift: {name}")
    counts = {k: len(v) for k, v in identity["candidates"].items()}
    if set(counts.values()) != {2 if smoke else 24} or set(counts) != {
        "cnn",
        "linear_svm",
        "logistic_regression",
        "random_forest",
        "xgboost",
    }:
        raise ValueError("Unexpected frozen candidate budget")
    if (
        identity["input_hz"] != 100
        or identity["input_samples"] != 6000
        or str(make_cnn()) != identity["architecture"]
    ):
        raise ValueError("Frozen waveform/architecture contract changed")
    if sum(p.numel() for p in make_cnn().parameters()) != 3009:
        raise ValueError("CNN parameter count drift")
    return payload, manifest["identity_hash"]


def event_log(path, event, **fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as stream:
        stream.write(
            json.dumps(
                dict(time_utc=datetime.now(UTC).isoformat(), event=event, **fields), default=str
            )
            + "\n"
        )


@contextmanager
def exclusive_run(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    with (directory / "execution.lock").open("a+") as stream:
        try:
            fcntl.flock(stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Evaluation already running in this directory") from exc
        try:
            yield
        finally:
            fcntl.flock(stream.fileno(), fcntl.LOCK_UN)


def checkpoint_task(path, context, compute):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    context_hash = hashlib.sha256(canonical(context).encode()).hexdigest()
    log = path.parent / "events.jsonl"
    if path.exists():
        try:
            manifest = json.loads((path / "manifest.json").read_text())
            if (
                manifest["context_hash"] != context_hash
                or digest_file(path / "payload.pkl") != manifest["payload_hash"]
            ):
                raise ValueError("Checkpoint context/content mismatch")
            with (path / "payload.pkl").open("rb") as stream:
                result = pickle.load(stream)
        except (KeyError, FileNotFoundError, json.JSONDecodeError) as exc:
            raise ValueError("Incomplete/corrupt checkpoint") from exc
        event_log(log, "resumed", task=path.name)
        return result
    event_log(log, "started", task=path.name)
    try:
        result = compute()
        with tempfile.TemporaryDirectory(prefix=".task-", dir=path.parent) as temporary:
            stage = Path(temporary) / "task"
            stage.mkdir()
            with (stage / "payload.pkl").open("wb") as stream:
                pickle.dump(result, stream, protocol=5)
            (stage / "manifest.json").write_text(
                json.dumps(
                    dict(
                        context_hash=context_hash,
                        context=context,
                        payload_hash=digest_file(stage / "payload.pkl"),
                    ),
                    indent=2,
                )
                + "\n"
            )
            stage.rename(path)
    except BaseException as exc:
        event_log(log, "failed", task=path.name, error_type=type(exc).__name__, error=str(exc))
        raise
    event_log(log, "completed", task=path.name)
    return result


def partition_indices(cohort, folds, stops, outer):
    if cohort.duplicated(["record_id", "minute"]).any():
        raise ValueError("Duplicate cohort keys")
    part = folds[folds.outer_fold.eq(outer)]
    if part.duplicated("record_id").any() or set(part.record_id) != set(cohort.record_id):
        raise ValueError("Incomplete/duplicate outer assignments")
    if not set(part.outer_role).issubset({"train", "test"}):
        raise ValueError("Invalid outer role")
    joined = cohort[["record_id", "subject_group", "label"]].merge(
        part, on=["record_id", "subject_group"], validate="many_to_one"
    )
    if len(joined) != len(cohort) or part.groupby("subject_group").outer_role.nunique().max() != 1:
        raise ValueError("Outer group leakage or identity mismatch")
    train = np.flatnonzero(cohort.record_id.isin(part.query("outer_role == 'train'").record_id))
    held = np.flatnonzero(cohort.record_id.isin(part.query("outer_role == 'test'").record_id))
    local = cohort.iloc[train].reset_index(drop=True)
    cv, calibration = [], []

    def require_two(indices):
        if set(local.iloc[indices].label) != {0, 1}:
            raise ValueError("Class-infeasible inner partition")

    inner_part = part.query("outer_role == 'train'")
    if inner_part.groupby("subject_group").inner_fold.nunique().max() != 1:
        raise ValueError("Inner group leakage")
    for inner in sorted(inner_part.inner_fold.unique()):
        valid = np.flatnonzero(
            local.record_id.isin(inner_part[inner_part.inner_fold.eq(inner)].record_id)
        )
        fit = np.setdiff1d(np.arange(len(local)), valid)
        stopping = stops[stops.outer_fold.eq(outer) & stops.inner_fold.eq(inner)]
        if stopping.duplicated("record_id").any() or set(stopping.record_id) != set(
            local.record_id
        ):
            raise ValueError("Incomplete/duplicate stopping assignments")
        if set(stopping.role) != {"calibration-fit", "calibration-stop", "inner-validation"}:
            raise ValueError("Invalid stopping roles")
        if stopping.groupby("subject_group").role.nunique().max() != 1:
            raise ValueError("Stopping group leakage")
        cal = np.flatnonzero(
            local.record_id.isin(stopping[stopping.role.eq("calibration-fit")].record_id)
        )
        stop = np.flatnonzero(
            local.record_id.isin(stopping[stopping.role.eq("calibration-stop")].record_id)
        )
        score = np.flatnonzero(
            local.record_id.isin(stopping[stopping.role.eq("inner-validation")].record_id)
        )
        if set(cal) & set(stop) or set(cal) | set(stop) != set(fit) or set(score) != set(valid):
            raise ValueError("Stopping/fitting/scoring overlap or missing coverage")
        for indices in [fit, valid, cal, stop]:
            require_two(indices)
        cv.append((fit, valid))
        calibration.append((cal, stop))
    if sorted(np.concatenate([v for _, v in cv]).tolist()) != list(range(len(train))):
        raise ValueError("Inner validation coverage must be exactly once")
    if set(cohort.iloc[held].label) != {0, 1}:
        raise ValueError("Class-infeasible outer test")
    return train, held, cv, calibration

## 2. Inner selection and fresh CNN calibration/refit

The selected model sees only inner fold scores. LR/SVM grids and RF/XGBoost randomized searches evaluate the exact persisted points; candidate IDs are remapped to frozen order so search execution order cannot change tie-breaking. Every preprocessing fit occurs within its training fold. All failures and warnings remain visible; convergence failures stop execution.

For each CNN candidate/inner fold: calibrate epochs on the separate stopping groups, discard that model for scoring, initialize afresh on all inner training groups for exactly the calibrated epochs, and score inner validation once. The final outer epoch count is round-half-up median calibrated epochs. Repeated seeds reuse selected settings and epoch schedules, with newly cross-fitted thresholds.

In [3]:
def select_candidate(trials):
    ordered = sorted(trials, key=lambda r: r["candidate"])
    if not ordered or not all(np.isfinite(r["mean_validation_ap"]) for r in ordered):
        raise ValueError("Missing/nonfinite candidate scores")
    best = ordered[0]
    for row in ordered[1:]:
        if row["mean_validation_ap"] > best["mean_validation_ap"] + 1e-12:
            best = row
    return best["candidate"]


def validate_learning_arrays(x, y, cv=None):
    if len(x) != len(y) or set(np.asarray(y)) != {0, 1} or np.isinf(np.asarray(x)).any():
        raise ValueError("Invalid fitting arrays/classes")
    splits = cv if cv is not None else [(np.arange(len(y)), np.array([], dtype=int))]
    for fit, valid in splits:
        if set(fit) & set(valid) or np.isnan(np.asarray(x)[fit]).all(axis=0).any():
            raise ValueError("Overlapping fold or all-missing fitting predictor")
        if set(np.asarray(y)[fit]) != {0, 1}:
            raise ValueError("Class-degenerate fitting fold")
        if len(valid) and set(np.asarray(y)[valid]) != {0, 1}:
            raise ValueError("Class-degenerate scoring fold")


def sklearn_search(name, x, y, cv, candidates, seed):
    validate_learning_arrays(x, y, cv)
    if name == "xgboost":
        return xgb_worker("search", dict(x=x, y=y, cv=cv, candidates=candidates, seed=seed))
    model = make_classical_model(name, {}, seed)
    prefix = model.steps[-1][0] + "__"
    points = [{prefix + k: [v] for k, v in c.items()} for c in candidates]
    common = dict(
        estimator=model,
        scoring="average_precision",
        cv=cv,
        refit=False,
        return_train_score=False,
        error_score="raise",
        n_jobs=1,
    )
    if name == "random_forest":
        search = RandomizedSearchCV(
            param_distributions=points, n_iter=len(points), random_state=1067, **common
        )
    else:
        search = GridSearchCV(param_grid=points, **common)
    start = perf_counter()
    with warnings.catch_warnings(record=True) as notices:
        warnings.simplefilter("always")
        search.fit(x, y)
    records = []
    for rank, params in enumerate(search.cv_results_["params"]):
        raw = {k.removeprefix(prefix): v for k, v in params.items()}
        candidate = next(i for i, c in enumerate(candidates) if canonical(c) == canonical(raw))
        records.append(
            dict(
                candidate=candidate,
                settings=raw,
                mean_validation_ap=float(search.cv_results_["mean_test_score"][rank]),
                split_ap=[
                    float(search.cv_results_[f"split{i}_test_score"][rank]) for i in range(len(cv))
                ],
                mean_fit_seconds=float(search.cv_results_["mean_fit_time"][rank]),
            )
        )
    if sorted(r["candidate"] for r in records) != list(range(len(candidates))):
        raise ValueError("Search did not evaluate every frozen candidate exactly once")
    notes = [dict(category=w.category.__name__, message=str(w.message)) for w in notices]
    if any(w["category"] == "ConvergenceWarning" for w in notes):
        raise RuntimeError(
            "Search convergence failure; inspect recorded settings before any protocol amendment"
        )
    return dict(
        trials=sorted(records, key=lambda r: r["candidate"]),
        warnings=notes,
        seconds=perf_counter() - start,
    )


def xgb_worker(operation, data):
    """All XGBoost fitting AND scoring happen in a torch-free interpreter."""
    worker = """import json, pickle, sys, time, warnings
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

operation, d = pickle.loads(sys.stdin.buffer.read())


def canonical(x):
    return json.dumps(x, sort_keys=True, default=lambda v: v.item())


def model(settings):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            scale_pos_weight=1,
            n_jobs=4,
            random_state=d["seed"],
            **settings,
        ),
    )


start = time.perf_counter()
with warnings.catch_warnings(record=True) as notices:
    warnings.simplefilter("always")
    if operation == "search":
        prefix = "xgbclassifier__"
        points = [{prefix + k: [v] for k, v in c.items()} for c in d["candidates"]]
        search = RandomizedSearchCV(
            model({}),
            points,
            n_iter=len(points),
            random_state=1067,
            cv=d["cv"],
            scoring="average_precision",
            refit=False,
            error_score="raise",
            return_train_score=False,
            n_jobs=1,
        )
        search.fit(d["x"], d["y"])
        trials = []
        for r, params in enumerate(search.cv_results_["params"]):
            raw = {k.removeprefix(prefix): v for k, v in params.items()}
            candidate = next(
                i for i, c in enumerate(d["candidates"]) if canonical(c) == canonical(raw)
            )
            trials.append(
                dict(
                    candidate=candidate,
                    settings=raw,
                    mean_validation_ap=float(search.cv_results_["mean_test_score"][r]),
                    split_ap=[
                        float(search.cv_results_[f"split{i}_test_score"][r])
                        for i in range(len(d["cv"]))
                    ],
                    mean_fit_seconds=float(search.cv_results_["mean_fit_time"][r]),
                )
            )
        result = dict(trials=sorted(trials, key=lambda r: r["candidate"]))
    elif operation == "fit":
        fitted = model(d["settings"])
        fitted.fit(d["x"], d["y"])
        result = dict(
            scores=fitted.predict_proba(d["score_x"])[:, 1],
            model_bytes=pickle.dumps(fitted, protocol=5),
        )
    else:
        raise ValueError("Unknown XGBoost operation")
result.update(
    warnings=[dict(category=w.category.__name__, message=str(w.message)) for w in notices],
    seconds=time.perf_counter() - start,
    torch_loaded="torch" in sys.modules,
)
sys.stdout.buffer.write(pickle.dumps(result, protocol=5))
"""
    completed = subprocess.run(
        [sys.executable, "-c", worker],
        input=pickle.dumps((operation, data), protocol=5),
        capture_output=True,
        timeout=7200,
        check=False,
    )
    if completed.returncode:
        raise RuntimeError(
            f"Isolated XGBoost failure ({completed.returncode}): "
            f"{completed.stderr.decode(errors='replace')}"
        )
    result = pickle.loads(completed.stdout)
    if result["torch_loaded"]:
        raise ValueError("XGBoost interpreter imported torch")
    if operation == "search" and sorted(r["candidate"] for r in result["trials"]) != list(
        range(len(data["candidates"]))
    ):
        raise ValueError("XGBoost frozen candidate coverage mismatch")
    return result


def fit_predict_classical(name, x, y, score_x, settings, seed):
    validate_learning_arrays(x, y)
    if name == "xgboost":
        return xgb_worker("fit", dict(x=x, y=y, score_x=score_x, settings=settings, seed=seed))
    parameters = dict(settings)
    iterations = parameters.pop("max_iter", None)
    model = make_classical_model(name, parameters, seed)
    if iterations is not None:
        model.set_params(**{model.steps[-1][0] + "__max_iter": iterations})
    start = perf_counter()
    with warnings.catch_warnings(record=True) as notices:
        warnings.simplefilter("always")
        model.fit(x, y)
    if any(issubclass(w.category, ConvergenceWarning) for w in notices):
        raise RuntimeError("Refit convergence failure")
    scores = (
        model.decision_function(score_x)
        if name == "linear_svm"
        else model.predict_proba(score_x)[:, 1]
    )
    return dict(
        model_bytes=pickle.dumps(model, protocol=5),
        scores=np.asarray(scores),
        seconds=perf_counter() - start,
        warnings=[dict(category=w.category.__name__, message=str(w.message)) for w in notices],
    )


def state_hash(model):
    return hashlib.sha256(
        b"".join(v.detach().cpu().numpy().tobytes() for v in model.state_dict().values())
    ).hexdigest()


def cnn_scores(model_bytes, x, batch_size=128):
    model = make_cnn()
    model.load_state_dict(
        torch.load(io.BytesIO(model_bytes), map_location="cpu", weights_only=True)
    )
    model.eval()
    chunks = []
    with torch.inference_mode():
        for start in range(0, len(x), batch_size):
            values = torch.as_tensor(
                np.array(x[start : start + batch_size], copy=True), dtype=torch.float32
            )
            chunks.append(torch.sigmoid(model(values).squeeze(1)).numpy())
    scores = np.concatenate(chunks)
    if scores.shape != (len(x),) or not np.isfinite(scores).all():
        raise ValueError("Invalid CNN scores")
    return scores


def cnn_fit(x, y, settings, seed, max_epochs, stop_data=None, patience=8, min_delta=0.0001):
    if (
        np.asarray(x).ndim != 3
        or np.asarray(x).shape[1:] != (1, 6000)
        or set(np.asarray(y)) != {0, 1}
    ):
        raise ValueError("Invalid CNN fitting shape/classes")
    if max_epochs < 1:
        raise ValueError("Training needs at least one epoch")
    reset_seed(seed)
    model = make_cnn()
    initial = state_hash(model)
    weight = (
        float(np.sum(np.asarray(y) == 0) / np.sum(np.asarray(y) == 1))
        if settings["pos_weight"] == "balanced"
        else 1.0
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight))
    validation_criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"]
    )
    loader = DataLoader(
        TensorDataset(
            torch.as_tensor(np.array(x, copy=True), dtype=torch.float32),
            torch.as_tensor(np.asarray(y), dtype=torch.float32),
        ),
        batch_size=settings["batch_size"],
        shuffle=True,
        num_workers=0,
    )
    best_loss, stale, best_epoch = float("inf"), 0, 0
    history = []
    best_state = None
    for epoch in range(1, max_epochs + 1):
        start = perf_counter()
        loss = train_epoch(model, loader, optimizer, criterion)
        row = dict(epoch=epoch, train_loss=loss, pos_weight=weight)
        if stop_data is not None:
            stop_x, stop_y = stop_data
            if set(np.asarray(stop_y)) != {0, 1}:
                raise ValueError("Class-degenerate stopping data")
            model.eval()
            total = 0.0
            with torch.inference_mode():
                for offset in range(0, len(stop_y), settings["batch_size"]):
                    batch = torch.as_tensor(
                        np.array(stop_x[offset : offset + settings["batch_size"]], copy=True),
                        dtype=torch.float32,
                    )
                    truth = torch.as_tensor(
                        np.asarray(stop_y[offset : offset + settings["batch_size"]]),
                        dtype=torch.float32,
                    )
                    total += float(validation_criterion(model(batch).squeeze(1), truth)) * len(
                        truth
                    )
            stop_loss = total / len(stop_y)
            row["stop_loss"] = stop_loss
            if not np.isfinite(stop_loss):
                raise ValueError("Nonfinite stopping loss")
            if stop_loss < best_loss - min_delta:
                best_loss, stale, best_epoch = stop_loss, 0, epoch
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            else:
                stale += 1
        else:
            best_epoch = epoch
        row["seconds"] = perf_counter() - start
        history.append(row)
        if stop_data is not None and stale >= patience:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    stream = io.BytesIO()
    torch.save(model.state_dict(), stream)
    return dict(
        model_bytes=stream.getvalue(),
        history=history,
        best_epoch=best_epoch,
        pos_weight=weight,
        initial_state_hash=initial,
        final_state_hash=state_hash(model),
    )


def run_inner_search(name, x, y, cv, stopping, candidates, rules, seed, path, context):
    if name != "cnn":
        return checkpoint_task(
            Path(path) / "search", context, lambda: sklearn_search(name, x, y, cv, candidates, seed)
        )
    validate_learning_arrays(x.reshape(len(x), -1), y, cv)
    trials = []
    for candidate, settings in enumerate(candidates):
        scores = []
        epochs = []
        for inner, ((fit, valid), (cal, stop)) in enumerate(zip(cv, stopping, strict=True)):
            if set(cal) & set(stop) or set(cal) | set(stop) != set(fit):
                raise ValueError("CNN stopping leakage")
            key = dict(context, candidate=candidate, inner=inner, settings=settings)

            def execute(cal=cal, stop=stop, fit=fit, valid=valid, settings=settings):
                calibrated = cnn_fit(
                    x[cal],
                    y[cal],
                    settings,
                    seed,
                    rules["cnn_max_epochs"],
                    stop_data=(x[stop], y[stop]),
                    patience=rules["cnn_patience"],
                    min_delta=rules["cnn_min_delta"],
                )
                refit = cnn_fit(x[fit], y[fit], settings, seed, calibrated["best_epoch"])
                prediction = cnn_scores(refit["model_bytes"], x[valid])
                return dict(
                    calibration=calibrated,
                    refit=refit,
                    scores=prediction,
                    validation_ap=float(average_precision_score(y[valid], prediction)),
                )

            result = checkpoint_task(
                Path(path) / f"candidate-{candidate:02d}-inner-{inner}", key, execute
            )
            scores.append(result["validation_ap"])
            epochs.append(result["calibration"]["best_epoch"])
        trials.append(
            dict(
                candidate=candidate,
                settings=settings,
                mean_validation_ap=float(np.mean(scores)),
                split_ap=scores,
                calibrated_epochs=epochs,
            )
        )
        print(
            f"{Path(path).name}: CNN candidate {candidate + 1}/{len(candidates)} complete",
            flush=True,
        )
    return dict(trials=trials, warnings=[])


def inner_oof(
    name, x, y, cv, settings, seed, path, context, epochs=None, search_path=None, candidate=None
):
    scores = np.full(len(y), np.nan)
    seen = np.zeros(len(y), dtype=int)
    for inner, (fit, valid) in enumerate(cv):
        key = dict(context, inner=inner, settings=settings)

        def execute(fit=fit, valid=valid, inner=inner):
            if name == "cnn":
                model = cnn_fit(x[fit], y[fit], settings, seed, int(epochs[inner]))
                return dict(**model, scores=cnn_scores(model["model_bytes"], x[valid]))
            return fit_predict_classical(name, x[fit], y[fit], x[valid], settings, seed)

        if name == "cnn" and search_path is not None:
            key = dict(context, candidate=candidate, inner=inner, settings=settings)
            # Primary CNN OOF comes from the selected saved candidate, never a second training run.
            task = Path(search_path) / f"candidate-{candidate:02d}-inner-{inner}"
            result = checkpoint_task(
                task, key, lambda: (_ for _ in ()).throw(ValueError("Missing selected CNN trial"))
            )
        else:
            result = checkpoint_task(Path(path) / f"inner-{inner}", key, execute)
        scores[valid] = result["scores"]
        seen[valid] += 1
    if not np.isfinite(scores).all() or not np.all(seen == 1):
        raise ValueError("Incomplete inner OOF scores")
    return scores


def fit_outer_model(name, x, y, score_x, settings, seed, path, context, epoch_count=None):
    def execute():
        if name == "cnn":
            model = cnn_fit(x, y, settings, seed, int(epoch_count))
            return dict(**model, scores=cnn_scores(model["model_bytes"], score_x))
        return fit_predict_classical(name, x, y, score_x, settings, seed)

    return checkpoint_task(path, dict(context, settings=settings, epoch_count=epoch_count), execute)

## 3. Complete comparison matrix, artifacts and reporting

For every outer fold: tune all five core pipelines, refit selected RF/XGBoost/CNN settings for both repeat seeds, run all 15 fixed-setting feature ablations, and add the three trivial references. Thresholds are selected exclusively from inner out-of-fold scores. Outer labels are joined only for stored evaluation rows, not passed to candidate selection or stopping.

Output includes candidate/fold scores, selected settings, fitted model bytes, training histories/warnings, inner/outer predictions, per-group/fold/pooled metrics, paired differences, conditional bootstrap intervals and an HTML inspection table. Full source notebooks stay output-free. These remain exploratory internal results with unverified patient identities and prior test exposure.

In [4]:
def prepare_inputs(run, root, freeze, output, context, phase1, feature_run):
    run, root, output = Path(run), Path(root), Path(output)
    cohort = pd.read_parquet(run / "protocol/cohort.parquet").reset_index(drop=True)
    folds = pd.read_csv(run / "protocol/fold_assignments.csv")
    stops = pd.read_csv(run / "protocol/cnn_stop_assignments.csv")
    config = json.loads((run / "protocol/protocol.json").read_text())
    validate_fold_assignments(
        cohort,
        folds,
        stops,
        **{k: config[k] for k in ["outer_seed", "outer_splits", "inner_splits", "stopping_splits"]},
    )
    names = ["mean_rr_ms", "median_rr_ms", "mean_hr_bpm", "sdrr_ms", "rmssd_ms", "prr50_pct"]
    smoke = freeze["identity"]["synthetic"]
    if smoke:
        full = np.random.default_rng(67).normal(size=(len(cohort), 6))
    else:
        for name, expected in freeze["identity"]["input_hashes"].items():
            path = (
                Path(feature_run) / "features.parquet"
                if name == "features"
                else Path(phase1) / name
            )
            if digest_file(path) != expected:
                raise ValueError(f"Frozen input changed: {name}")
        raw = pd.read_parquet(Path(feature_run) / "features.parquet")
        joined = cohort[["record_id", "minute", "label"]].merge(
            raw[["record_id", "minute", "label", *names]],
            on=["record_id", "minute"],
            validate="one_to_one",
            suffixes=("", "_source"),
        )
        if len(joined) != len(cohort) or not joined.label.equals(joined.label_source):
            raise ValueError("Feature label/key mismatch")
        full = joined[names].to_numpy(dtype=float)
    baseline = pd.read_parquet(run / "baselines/features.parquet")
    joined = cohort[["record_id", "minute", "label"]].merge(
        baseline, on=["record_id", "minute"], validate="one_to_one", suffixes=("", "_source")
    )
    if len(joined) != len(cohort) or not joined.label.equals(joined.label_source):
        raise ValueError("Literature key/label mismatch")
    specification = json.loads((run / "baselines/specifications.json").read_text())
    if specification["feature_order"] != literature_feature_names():
        raise ValueError("Literature feature-order drift")
    matrices = dict(f2=full, linear_svm=joined[literature_feature_names()].to_numpy(dtype=float))
    cache = output / "inputs"
    if cache.exists():
        metadata = json.loads((cache / "manifest.json").read_text())
        if (
            metadata["context"] != context
            or digest_file(cache / "waveforms.npy") != metadata["sha256"]
        ):
            raise ValueError("Waveform cache identity/content mismatch")
    else:
        with tempfile.TemporaryDirectory(prefix=".inputs-", dir=output) as temporary:
            stage = Path(temporary) / "inputs"
            stage.mkdir()
            array = np.lib.format.open_memmap(
                stage / "waveforms.npy", mode="w+", dtype="float32", shape=(len(cohort), 6000)
            )
            if smoke:
                rng = np.random.default_rng(67)
                for i in range(len(cohort)):
                    array[i] = normalize_minute(rng.normal(size=6000), 100)
            else:
                for record, part in cohort.groupby("record_id"):
                    with np.load(Path(phase1) / "waveforms" / f"{record}.npz") as source:
                        signal = source["filtered"]
                        fs = int(source["fs"])
                        for row in part.itertuples():
                            array[row.Index] = normalize_minute(
                                signal[row.start_sample : row.end_sample], fs
                            )
            array.flush()
            del array
            (stage / "manifest.json").write_text(
                json.dumps(
                    dict(context=context, sha256=digest_file(stage / "waveforms.npy")), indent=2
                )
                + "\n"
            )
            stage.rename(cache)
    waveforms = np.load(cache / "waveforms.npy", mmap_mode="r")
    if waveforms.shape != (len(cohort), 6000) or waveforms.dtype != np.float32:
        raise ValueError("Invalid waveform cache shape/type")
    matrices["cnn"] = waveforms[:, None, :]
    return cohort, folds, stops, matrices, names


def prediction_rows(
    cohort, indices, scores, threshold, outer, model, feature_set, analysis, seed, score_kind
):
    if np.asarray(scores).shape != (len(indices),) or not np.isfinite(scores).all():
        raise ValueError("Invalid score coverage")
    result = (
        cohort.iloc[indices][["record_id", "minute", "subject_group", "label"]]
        .copy()
        .reset_index(drop=True)
    )
    result["outer_fold"] = outer
    result["model"] = model
    result["feature_set"] = feature_set
    result["analysis"] = analysis
    result["seed"] = seed
    result["score"] = scores
    result["score_kind"] = score_kind
    result["threshold"] = threshold
    result["prediction"] = (np.asarray(scores) >= threshold).astype(int)
    return result


def validate_outer_predictions(table, cohort, folds):
    groups = ["analysis", "model", "feature_set", "seed"]
    key = ["record_id", "minute"]
    expected = cohort[key + ["subject_group", "label"]].merge(
        folds.query("outer_role == 'test'")[["record_id", "outer_fold"]],
        on="record_id",
        validate="many_to_one",
    )
    expected = expected.sort_values(key).reset_index(drop=True)
    for _, part in table.groupby(groups, dropna=False):
        validate_predictions(part)
        actual = (
            part[key + ["subject_group", "label", "outer_fold"]]
            .sort_values(key)
            .reset_index(drop=True)
        )
        pd.testing.assert_frame_equal(actual, expected, check_dtype=False)
        if not np.array_equal(
            part.prediction.to_numpy(), (part.score >= part.threshold).to_numpy().astype(int)
        ):
            raise ValueError("Predictions disagree with stored thresholds")


def run_evaluation(cohort, folds, stops, matrices, names, freeze, output, context):
    output = Path(output)
    jobs = output / "checkpoints"
    jobs.mkdir(exist_ok=True)
    candidates = freeze["identity"]["candidates"]
    rules = freeze["identity"]["rules"]
    all_predictions = []
    all_oof = []
    all_trials = []
    choices = []
    feature_sets = {
        "A_F1": names[:3],
        "B_F2": names,
        "C_no_SDRR": [n for n in names if n != "sdrr_ms"],
        "D_no_RMSSD": [n for n in names if n != "rmssd_ms"],
        "E_no_pRR50": [n for n in names if n != "prr50_pct"],
    }
    for outer in sorted(folds.outer_fold.unique()):
        train, held, cv, stopping = partition_indices(cohort, folds, stops, outer)
        y = cohort.iloc[train].label.to_numpy()
        effective_seed = 67 + 10000 * int(outer)
        for name in ["logistic_regression", "random_forest", "xgboost", "linear_svm", "cnn"]:
            base = matrices[name] if name in ["linear_svm", "cnn"] else matrices["f2"]
            x, score_x = base[train], base[held]
            path = jobs / f"outer-{outer}" / "tuned" / name
            search_context = dict(
                context, outer=int(outer), analysis="tuned", model=name, seed=67, operation="search"
            )
            search = run_inner_search(
                name,
                x,
                y,
                cv,
                stopping,
                candidates[name],
                rules,
                effective_seed,
                path,
                search_context,
            )
            candidate = select_candidate(search["trials"])
            settings = candidates[name][candidate]
            selected = next(t for t in search["trials"] if t["candidate"] == candidate)
            epochs = selected.get("calibrated_epochs")
            epoch_count = (
                int(np.clip(np.floor(np.median(epochs) + 0.5), 1, rules["cnn_max_epochs"]))
                if epochs
                else None
            )
            for trial in search["trials"]:
                all_trials.append(
                    dict(
                        outer_fold=int(outer),
                        model=name,
                        seed=67,
                        analysis="tuned",
                        candidate=trial["candidate"],
                        settings=canonical(trial["settings"]),
                        mean_validation_ap=trial["mean_validation_ap"],
                        split_ap=json.dumps(trial["split_ap"]),
                    )
                )
            seeds = [67] + (
                rules["repeat_seeds"] if name in ["random_forest", "xgboost", "cnn"] else []
            )
            for seed in seeds:
                job_context = dict(
                    context,
                    outer=int(outer),
                    analysis="tuned",
                    model=name,
                    seed=int(seed),
                    candidate=int(candidate),
                )
                primary_cnn = name == "cnn" and seed == 67
                oof = inner_oof(
                    name,
                    x,
                    y,
                    cv,
                    settings,
                    seed + 10000 * int(outer),
                    path / f"seed-{seed}" / "oof",
                    search_context if primary_cnn else dict(job_context, operation="oof"),
                    epochs=epochs,
                    search_path=path if primary_cnn else None,
                    candidate=candidate,
                )
                threshold = choose_threshold(y, oof)
                kind = "decision_function" if name == "linear_svm" else "probability"
                feature = (
                    "local_rr_11_adapted"
                    if name == "linear_svm"
                    else "waveform_100hz"
                    if name == "cnn"
                    else "B_F2"
                )
                inner_rows = prediction_rows(
                    cohort, train, oof, threshold, outer, name, feature, "tuned", seed, kind
                )
                inner_rows["inner_fold"] = -1
                for inner, (_, valid) in enumerate(cv):
                    inner_rows.loc[valid, "inner_fold"] = inner
                all_oof.append(inner_rows)
                fitted = fit_outer_model(
                    name,
                    x,
                    y,
                    score_x,
                    settings,
                    seed + 10000 * int(outer),
                    path / f"seed-{seed}" / "outer",
                    dict(job_context, operation="outer-refit"),
                    epoch_count,
                )
                all_predictions.append(
                    prediction_rows(
                        cohort,
                        held,
                        fitted["scores"],
                        threshold,
                        outer,
                        name,
                        feature,
                        "tuned",
                        seed,
                        kind,
                    )
                )
                choices.append(
                    dict(
                        outer_fold=int(outer),
                        analysis="tuned",
                        model=name,
                        feature_set=feature,
                        seed=int(seed),
                        candidate=int(candidate),
                        settings=settings,
                        threshold=threshold,
                        inner_mean_ap=selected["mean_validation_ap"],
                        calibrated_epochs=epochs,
                        outer_epochs=epoch_count,
                    )
                )
                print(f"Outer {outer}: {name} seed {seed} refit complete", flush=True)
        for name, settings in rules["rq1_fixed"].items():
            for feature, columns in feature_sets.items():
                base = matrices["f2"][:, [names.index(n) for n in columns]]
                x, score_x = base[train], base[held]
                path = jobs / f"outer-{outer}" / "ablation" / name / feature
                job_context = dict(
                    context,
                    outer=int(outer),
                    analysis="ablation",
                    model=name,
                    feature_set=feature,
                    seed=67,
                )
                oof = inner_oof(
                    name,
                    x,
                    y,
                    cv,
                    settings,
                    effective_seed,
                    path / "oof",
                    dict(job_context, operation="oof"),
                )
                threshold = choose_threshold(y, oof)
                inner_rows = prediction_rows(
                    cohort,
                    train,
                    oof,
                    threshold,
                    outer,
                    name,
                    feature,
                    "ablation",
                    67,
                    "probability",
                )
                inner_rows["inner_fold"] = -1
                for inner, (_, valid) in enumerate(cv):
                    inner_rows.loc[valid, "inner_fold"] = inner
                all_oof.append(inner_rows)
                fitted = fit_outer_model(
                    name,
                    x,
                    y,
                    score_x,
                    settings,
                    effective_seed,
                    path / "outer",
                    dict(job_context, operation="outer-refit"),
                )
                all_predictions.append(
                    prediction_rows(
                        cohort,
                        held,
                        fitted["scores"],
                        threshold,
                        outer,
                        name,
                        feature,
                        "ablation",
                        67,
                        "probability",
                    )
                )
                choices.append(
                    dict(
                        outer_fold=int(outer),
                        analysis="ablation",
                        model=name,
                        feature_set=feature,
                        seed=67,
                        settings=settings,
                        threshold=threshold,
                    )
                )
            print(f"Outer {outer}: fixed {name} ablations complete", flush=True)
        evaluation = cohort.iloc[held].copy()
        evaluation["outer_fold"] = int(outer)
        references = constant_references(y, evaluation)
        references["analysis"] = "reference"
        all_predictions.append(references)
    predictions = pd.concat(all_predictions, ignore_index=True)
    validate_outer_predictions(predictions, cohort, folds)
    # Exact planned matrix: 5 core + 6 repeated-seed + 15 fixed ablations + 3 trivial references.
    if len(predictions) != len(cohort) * 29:
        raise ValueError("Incomplete model/feature/seed comparison matrix")
    return dict(
        predictions=predictions,
        oof=pd.concat(all_oof, ignore_index=True),
        trials=pd.DataFrame(all_trials),
        choices=choices,
    )


def summarize_comparison(predictions):
    group_columns = ["analysis", "model", "feature_set", "seed"]
    rows = []
    for key, part in predictions.groupby(group_columns):
        result = summarize_predictions(part)
        for column, value in zip(group_columns, key, strict=True):
            result[column] = value
        rows.append(result)
    metrics = pd.concat(rows, ignore_index=True)
    paired = []
    tuned = predictions.query("analysis == 'tuned' and seed == 67")
    reference = tuned[tuned.model.eq("xgboost")]
    for name, part in tuned.groupby("model"):
        if name == "xgboost":
            continue
        for outer in sorted(part.outer_fold.unique()):
            left = reference[reference.outer_fold.eq(outer)]
            right = part[part.outer_fold.eq(outer)]
            paired.append(
                dict(
                    reference="xgboost",
                    comparison=name,
                    outer_fold=int(outer),
                    ap_difference=average_precision_score(left.label, left.score)
                    - average_precision_score(right.label, right.score),
                )
            )
    return metrics, pd.DataFrame(paired)


def export_results(output, result, context, freeze):
    output = Path(output)
    metrics, paired = summarize_comparison(result["predictions"])
    # These are conditional intervals for pooled predictions, distinct from fold means.
    tuned = result["predictions"].query("analysis == 'tuned' and seed == 67")
    reference = tuned[tuned.model.eq("xgboost")]
    intervals = []
    for name, part in tuned.groupby("model"):
        if name != "xgboost":
            intervals.append(
                dict(
                    reference="xgboost", comparison=name, **paired_group_bootstrap(reference, part)
                )
            )
    target = output / "results"
    with tempfile.TemporaryDirectory(prefix=".results-", dir=output) as temporary:
        stage = Path(temporary) / "results"
        stage.mkdir()
        result["predictions"].to_parquet(stage / "outer_predictions.parquet", index=False)
        result["oof"].to_parquet(stage / "inner_oof_predictions.parquet", index=False)
        result["trials"].to_parquet(stage / "trial_results.parquet", index=False)
        (stage / "selected_configurations.json").write_text(
            json.dumps(result["choices"], indent=2) + "\n"
        )
        metrics.to_csv(stage / "metrics.csv", index=False)
        paired.to_csv(stage / "paired_differences.csv", index=False)
        pd.DataFrame(intervals).to_csv(stage / "uncertainty.csv", index=False)
        # Export model bytes without loading native XGBoost objects in this process.
        model_dir = stage / "models"
        model_dir.mkdir()
        histories = []
        warnings_rows = []
        for checkpoint in sorted((output / "checkpoints").rglob("manifest.json")):
            metadata = json.loads(checkpoint.read_text())
            payload_path = checkpoint.parent / "payload.pkl"
            if digest_file(payload_path) != metadata["payload_hash"]:
                raise ValueError("Checkpoint changed during export")
            with payload_path.open("rb") as stream:
                payload = pickle.load(stream)
            key = str(checkpoint.parent.relative_to(output / "checkpoints"))
            for warning in payload.get("warnings", []):
                warnings_rows.append(dict(task=key, **warning))
            for section in ["calibration", "refit"]:
                for row in payload.get(section, {}).get("history", []):
                    histories.append(dict(task=key, stage=section, **row))
            for row in payload.get("history", []):
                histories.append(dict(task=key, stage="fit", **row))
            if checkpoint.parent.name in ["outer", "outer-refit"] and "model_bytes" in payload:
                name = key.replace("/", "__") + (
                    ".pt" if metadata["context"].get("model") == "cnn" else ".pkl"
                )
                (model_dir / name).write_bytes(payload["model_bytes"])
        pd.DataFrame(histories).to_parquet(stage / "training_history.parquet", index=False)
        (stage / "fit_warnings.json").write_text(json.dumps(warnings_rows, indent=2) + "\n")
        headline = metrics.query("level == 'fold_mean'")
        html = (
            "<h1>Nested evaluation — "
            + ("SYNTHETIC SMOKE" if context["synthetic"] else "exploratory retrospective")
            + "</h1>"
        )
        html += (
            "<p>Original patient identities unverified. Prior test outcomes were inspected. "
            "Fold means and pooled metrics are separate.</p>"
        )
        html += headline.to_html(index=False)
        (stage / "summary.html").write_text(html)
        artifacts = {
            str(p.relative_to(stage)): digest_file(p) for p in stage.rglob("*") if p.is_file()
        }
        (stage / "manifest.json").write_text(
            json.dumps(
                dict(
                    context=context,
                    artifacts=artifacts,
                    synthetic=context["synthetic"],
                    status="complete",
                    freeze_status=freeze["status"],
                ),
                indent=2,
            )
            + "\n"
        )
        if target.exists():
            raise ValueError("Results already exist; validate/resume rather than overwrite")
        stage.rename(target)
    return metrics


def verify_results(path, context):
    path = Path(path)
    metadata = json.loads((path / "manifest.json").read_text())
    if metadata["context"] != context or metadata["status"] != "complete":
        raise ValueError("Results context mismatch")
    for name, expected in metadata["artifacts"].items():
        if digest_file(path / name) != expected:
            raise ValueError(f"Result corruption: {name}")
    return pd.read_csv(path / "metrics.csv")

## 4. Verify, execute or resume

A new runner hash requires a new tag; an unchanged run resumes completed tasks without fitting them again. Real `verify` mode checks all dependencies, input hashes, class support, saved fold coverage and output identities without creating model fits. `summarize` verifies completed exports and performs no training. A failed task leaves no completed checkpoint and may be retried after correcting the underlying problem; incompatible completed tasks must never be overwritten.

In [5]:
install_dependencies(ROOT, globals())
make_cnn = globals()["make_cnn"]
normalize_minute = globals()["normalize_minute"]
make_classical_model = globals()["make_classical_model"]
reset_seed = globals()["reset_seed"]
train_epoch = globals()["train_epoch"]
choose_threshold = globals()["choose_threshold"]
validate_predictions = globals()["validate_predictions"]
metric_values = globals()["metric_values"]
summarize_predictions = globals()["summarize_predictions"]
group_weights = globals()["group_weights"]
paired_group_bootstrap = globals()["paired_group_bootstrap"]
validate_cohort = globals()["validate_cohort"]
require_classes = globals()["require_classes"]
build_fold_assignments = globals()["build_fold_assignments"]
validate_fold_assignments = globals()["validate_fold_assignments"]
constant_references = globals()["constant_references"]
literature_feature_names = globals()["literature_feature_names"]
freeze, freeze_id = verify_freeze(RUN, ROOT, SMOKE)
context = dict(
    freeze_id=freeze_id,
    runner_sha256=digest_file(ROOT / "notebooks/08_nested_evaluation.ipynb"),
    synthetic=SMOKE,
    version=1,
)
OUTPUT = RUN / "full" / TAG
print("Action:", ACTION, "Synthetic:", SMOKE, "Freeze:", freeze_id, "Output:", OUTPUT)
if ACTION == "verify":
    cohort = pd.read_parquet(RUN / "protocol/cohort.parquet")
    folds = pd.read_csv(RUN / "protocol/fold_assignments.csv")
    stops = pd.read_csv(RUN / "protocol/cnn_stop_assignments.csv")
    for outer in sorted(folds.outer_fold.unique()):
        partition_indices(cohort, folds, stops, outer)
    if not SMOKE:
        for name, expected in freeze["identity"]["input_hashes"].items():
            path = FEATURE_RUN / "features.parquet" if name == "features" else PHASE1 / name
            if digest_file(path) != expected:
                raise ValueError(f"Frozen input drift: {name}")
    print(
        "Preflight passed. No fitting performed. Candidates per model:",
        {k: len(v) for k, v in freeze["identity"]["candidates"].items()},
    )
elif ACTION == "summarize":
    metrics = verify_results(OUTPUT / "results", context)
    display(metrics.query("level == 'fold_mean'"))
else:
    with exclusive_run(OUTPUT):
        if (OUTPUT / "results").exists():
            metrics = verify_results(OUTPUT / "results", context)
            print("Verified complete run: no models retrained")
        else:
            print("Preparing frozen inputs...", flush=True)
            cohort, folds, stops, matrices, names = prepare_inputs(
                RUN, ROOT, freeze, OUTPUT, context, PHASE1, FEATURE_RUN
            )
            result = checkpoint_task(
                OUTPUT / "evaluation",
                context,
                lambda: run_evaluation(
                    cohort, folds, stops, matrices, names, freeze, OUTPUT, context
                ),
            )
            metrics = export_results(OUTPUT, result, context, freeze)
        display(metrics.query("level == 'fold_mean'"))
        print("Completed synthetic smoke" if SMOKE else "Completed exploratory nested evaluation")

Action: full Synthetic: False Freeze: 1765caa330e2136b1f16f61828fae0a568bcc36b0730bd3b2cfd9b3d7e2f1363 Output: <project-root>/data/processed/portfolio-v2/plan001-real-20261108/full/real-20261109-v1
Preparing frozen inputs...


Outer 0: logistic_regression seed 67 refit complete


Outer 0: random_forest seed 67 refit complete


Outer 0: random_forest seed 167 refit complete


Outer 0: random_forest seed 267 refit complete


Outer 0: xgboost seed 67 refit complete


... (183 outputs left out)

Outer 4: fixed random_forest ablations complete


Outer 4: fixed xgboost ablations complete


,level,identifier,n_windows,prevalence,tn,fp,fn,tp,sensitivity,specificity,...,f1_score,accuracy,average_precision,roc_auc,ranking_reason,fold_ap_std,analysis,model,feature_set,seed
64,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.401319,NaN,NaN,0.062138,ablation,logistic_regression,A_F1,67
129,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.649725,NaN,NaN,0.137042,ablation,logistic_regression,B_F2,67
194,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.356776,NaN,NaN,0.083562,ablation,logistic_regression,C_no_SDRR,67
259,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.520748,NaN,NaN,0.148266,ablation,logistic_regression,D_no_RMSSD,67
324,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.667813,NaN,NaN,0.132230,ablation,logistic_regression,E_no_pRR50,67
389,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.400268,NaN,NaN,0.119471,ablation,random_forest,A_F1,67
454,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.506840,NaN,NaN,0.105692,ablation,random_forest,B_F2,67
519,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.355201,NaN,NaN,0.050287,ablation,random_forest,C_no_SDRR,67
584,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.488830,NaN,NaN,0.110774,ablation,random_forest,D_no_RMSSD,67
649,fold_mean,primary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,0.519138,NaN,NaN,0.105461,ablation,random_forest,E_no_pRR50,67


Completed exploratory nested evaluation
